In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import os
import sys
sys.path.insert(0, "/".join(["..", "python"]))

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import rasterio
from rasterio.io import MemoryFile
from rasterio.transform import Affine, from_origin, xy

from gridr.chain.fft_filtering_chain import fft_filtering_oa_strip_chain
from gridr.core.convolution.fft_filtering import (
    decimated_size,
    decimation_offset,
    fft_array_filter,
    fft_array_filter_output_shape,
)
from gridr.misc.mandrill import mandrill
from notebook_utils import plot_im, mpl_plot_wrapper
IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
import notebook_utils
notebook_utils.FORCE_MPL_INTERACTIVE = True
if not IN_DOC_BUILD and not notebook_utils.FORCE_MPL_INTERACTIVE:
    from bokeh.io import output_notebook  # enables plot interface in J notebook
    output_notebook()


def gaussian_kernel_2d(shape, sigma):
    """
    Generates a normalized 2D Gaussian kernel.

    :param shape: Kernel size (int for a square kernel, or tuple (height, width))
    :param sigma: Standard deviation of the Gaussian (float or tuple (sigma_y, sigma_x))
    :return: a tuple containing:
        - 1D Numpy array representing x coordinates
        - 1D Numpy array representing y coordinates
        - 2D NumPy array representing the kernel
    """
    # Handle scalar or tuple inputs for shape and sigma
    if isinstance(shape, int):
        h = w = shape
    else:
        h, w = shape

    if isinstance(sigma, (int, float)):
        sigma_y = sigma_x = sigma
    else:
        sigma_y, sigma_x = sigma

    # Create centered coordinates (e.g., size 5 -> [-2, -1, 0, 1, 2])
    y = np.arange(-(h // 2), h // 2 + 1 if h % 2 != 0 else h // 2)
    x = np.arange(-(w // 2), w // 2 + 1 if w % 2 != 0 else w // 2)

    # 2D Grid
    x_grid, y_grid = np.meshgrid(x, y)

    # Calculate 2D Gaussian function
    kernel = np.exp(-0.5 * ((x_grid / sigma_x) ** 2 + (y_grid / sigma_y) ** 2))

    # Normalize so the sum of all elements equals 1
    kernel_sum = np.sum(kernel)
    if kernel_sum != 0:
        kernel /= kernel_sum

    return x, y, kernel


@mpl_plot_wrapper
def plot_output_geometry(cases=((3, "centered", (2, 3), (11, 11)),
                                (4, "centered", (2, 3), (11, 11)),
                                (4, "leading", (2, 3), (11, 11)),
                                (4, "centered", (4, 4), (11, 11))),
                         n=18, dpi=100, prefix=None):
    """Show the decimated output grid and its extent, in the input raster frame.

    Blue carries what the call produces: the output pixels as `output_transform`
    places them, the samples they hold, the corner the transform returns for
    output pixel (0, 0), and the extent of the output dataset. Grey carries the
    Q x Q blocks and their centers, drawn as a reference so that the decimation
    phase can be read off the figure. The phase and the output size come from
    `decimation_offset` and `decimated_size`, so the figure follows the
    implementation rather than restating it.

    :param cases: sequence of (decimation factor, decimation mode, (row, col)
        origin of the production window, (height, width) of that window)
    :param n: size of the displayed region, in input pixels
    :param dpi: resolution of the figure
    :return: the matplotlib figure
    """
    ncols = 2 if len(cases) > 3 else len(cases)
    nrows = int(np.ceil(len(cases) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 4.4 * nrows), dpi=dpi)
    axes = np.atleast_1d(axes).ravel()
    for extra in axes[len(cases):]:
        extra.set_axis_off()

    for ax, (q, mode, win_origin, win_shape) in zip(axes, cases):
        r0, c0 = win_origin
        nr, nc = win_shape
        offset = decimation_offset(q, mode)
        shift = offset + 0.5 - q / 2
        out_nr = decimated_size(nr, q, offset)
        out_nc = decimated_size(nc, q, offset)

        # Full resolution pixels, the window ones standing out
        for i in range(n):
            for j in range(n):
                inside = r0 <= i < r0 + nr and c0 <= j < c0 + nc
                ax.add_patch(Rectangle((j, i), 1, 1,
                                       facecolor="0.96" if inside else "0.86",
                                       edgecolor="0.9", linewidth=0.4))

        # Reference only: Q x Q blocks and their centers, inside the window
        by = r0 + q * np.arange(int(np.ceil(nr / q)))
        bx = c0 + q * np.arange(int(np.ceil(nc / q)))
        for y in by:
            for xx in bx:
                ax.add_patch(Rectangle((xx, y), min(q, c0 + nc - xx),
                                       min(q, r0 + nr - y), fill=False, zorder=2,
                                       edgecolor="0.6", linewidth=0.9,
                                       linestyle=(0, (2, 3))))
        gx, gy = np.meshgrid(bx + q / 2, by + q / 2)
        ax.plot(gx.ravel(), gy.ravel(), linestyle="none", marker="+", markersize=6,
                color="0.45", markeredgewidth=1.0, zorder=3)

        # Production window
        ax.add_patch(Rectangle((c0, r0), nc, nr, fill=False, zorder=4,
                               edgecolor="tab:orange", linewidth=1.8))

        # What the call produces
        ys = r0 + shift + q * np.arange(out_nr)
        xs = c0 + shift + q * np.arange(out_nc)
        for sy in ys:
            for sx in xs:
                first = np.isclose(sx, xs[0]) and np.isclose(sy, ys[0])
                ax.add_patch(Rectangle(
                    (sx, sy), q, q, zorder=5,
                    facecolor=(0.12, 0.47, 0.71, 0.22 if first else 0.06),
                    edgecolor="tab:blue", linewidth=2.0 if first else 1.3))
        gx, gy = np.meshgrid(c0 + offset + q * np.arange(out_nc) + 0.5,
                             r0 + offset + q * np.arange(out_nr) + 0.5)
        ax.plot(gx.ravel(), gy.ravel(), linestyle="none", marker="s", markersize=5,
                markerfacecolor="tab:blue", markeredgecolor="white",
                markeredgewidth=0.8, zorder=7)

        # Extent of the output dataset
        ax.add_patch(Rectangle((xs[0], ys[0]), out_nc * q, out_nr * q, fill=False,
                               zorder=8, edgecolor="tab:blue", linewidth=2.2,
                               linestyle=(0, (5, 2))))
        ax.plot([xs[0]], [ys[0]], marker="o", markersize=6, markerfacecolor="white",
                markeredgecolor="tab:blue", markeredgewidth=1.8, zorder=9)

        def side(d, where):
            if np.isclose(d, 0):
                return f"{where} on the window"
            return (f"{where} {abs(d):.1f} px outside" if d > 0
                    else f"{where} {abs(d):.1f} px inside")

        over_first = -shift                          # output extent to window origin
        over_last = xs[0] + out_nc * q - (c0 + nc)   # output extent past window end
        ax.set_title(f'Q = {q}, "{mode}", win origin ({r0}, {c0})\n'
                     f'offset = {offset}, shift = {shift:+.1f}', fontsize=9)
        ax.text(0.5, -0.155,
                f"window {nr}x{nc} px  ->  output {out_nr}x{out_nc} px",
                transform=ax.transAxes, ha="center", fontsize=8, color="0.25")
        ax.text(0.5, -0.225,
                f"extent {side(over_first, 'starts')}, {side(over_last, 'ends')}",
                transform=ax.transAxes, ha="center", fontsize=7.5, color="0.35")
        ax.set_xlim(0, n)
        ax.set_ylim(n, 0)  # image convention: rows downwards
        ax.set_aspect("equal")
        ticks_c = sorted({0, c0, c0 + nc, *bx})
        ticks_r = sorted({0, r0, r0 + nr, *by})
        ax.set_xticks([t for t in ticks_c if t <= n])
        ax.set_yticks([t for t in ticks_r if t <= n])
        ax.tick_params(labelsize=7)
        ax.set_xlabel("input raster column", fontsize=8)
        ax.set_ylabel("input raster row", fontsize=8)

    fig.text(0.5, 0.034, "blue: output pixels, kept samples, the corner the transform "
             "returns, and the output extent (dashed)",
             ha="center", fontsize=7.5, color="tab:blue")
    fig.text(0.5, 0.012, "grey: Q x Q blocks and their centers, for reference    "
             "orange: production window",
             ha="center", fontsize=7.5, color="0.45")
    fig.tight_layout(rect=[0, 0.055 if nrows > 1 else 0.13, 1, 1], h_pad=3.2)
    return fig

mandrill_0 = mandrill[0]

# The `gaussian_kernel_2d` function implements the formula given in the first tutorial.
gaussian_sigma = 4.
x, y, kernel = gaussian_kernel_2d(shape=int(gaussian_sigma * 6 + 1), sigma=gaussian_sigma)

# Production window and decimation on a raster

This second chain tutorial restricts the processing to a rectangular region of the input
raster, then decimates the result. Both arguments keep the meaning they have in the core
API, described in *Limiting computation to a window* and *Zooming out and aliasing*. What
changes here is everything that surrounds them: the strips are cut over the window, the
reads are restricted to it, and the output dataset has to be sized and placed
accordingly.

**What you'll learn**

- How ``win`` selects the production window, and what the chain reads because of it
- How to size the output dataset for a window and for a decimation
- How to georeference it, and what the decimation phase does to the output grid
- Why the phase does not depend on ``strip_size``

## Setting things up

The input raster is built in memory, as in the previous tutorial. The mandrill image is
512x512, which leaves room for a window well inside the raster.

In [ ]:
import numpy as np
import rasterio
from rasterio.io import MemoryFile
from rasterio.transform import Affine, from_origin, xy

from gridr.chain.fft_filtering_chain import fft_filtering_oa_strip_chain
from gridr.core.convolution.fft_filtering import (
    decimation_offset,
    fft_array_filter,
    fft_array_filter_output_shape,
)
from gridr.misc.mandrill import mandrill

mandrill_0 = mandrill[0]

profile_in = {
    "driver": "GTiff",
    "height": mandrill_0.shape[0],
    "width": mandrill_0.shape[1],
    "count": 1,
    "dtype": mandrill_0.dtype,
    "crs": "EPSG:32631",
    "transform": from_origin(400000.0, 5000000.0, 1.0, 1.0),
}

memfile_in = MemoryFile()
with memfile_in.open(**profile_in) as ds:
    ds.write(mandrill_0, 1)

ds_in = memfile_in.open()
ds_in.height, ds_in.width

Three output datasets are opened in this tutorial. The two helpers below keep the
examples readable: one opens an in-memory dataset with a given shape and transform, the
other closes it and reads the result back.

In [ ]:
def open_output(shape, transform, dtype="float32"):
    """Open an in-memory dataset sized and georeferenced for the filtering."""
    memfile = MemoryFile()
    profile = profile_in | {
        "height": shape[0],
        "width": shape[1],
        "dtype": dtype,
        "transform": transform,
    }
    return memfile, memfile.open(**profile)


def read_back(memfile, ds_out):
    """Flush the output dataset and read its single band back."""
    ds_out.close()
    with memfile.open() as src:
        return src.read(1)

## The production window

`win` gives the region to produce, as inclusive `(first, last)` bounds on each axis, in
the frame of the input raster. The chain cuts that window into strips, and not the
raster, and each read covers the strip extended by the kernel margins. A small window on
a large raster therefore costs a small read rather than a full pass over the file.

The boundary condition behaves as in the core layer. It applies where the margin leaves
the raster, so a window sitting in the interior is filtered from real neighbours on every
side and gives the same result under every policy. The window used here starts at row 0,
which is also the first row of the raster, so `boundary` does come into play along that
edge.

In [ ]:
win = np.asarray([[0, 191], [160, 351]])
boundary = "reflect"
out_mode = "same"

out_shape = fft_array_filter_output_shape(
    (ds_in.height, ds_in.width),
    kernel,
    win=win,
    boundary=boundary,
    out_mode=out_mode,
)
out_shape

The shape follows the window rather than the raster. The transform has to follow it as
well: the chain writes pixel values and never touches the georeferencing, so the caller
places the output grid. Without decimation and with `out_mode="same"`, the output starts
on the first pixel of the window, which is a plain translation of the input transform.

In [ ]:
row_first, col_first = int(win[0, 0]), int(win[1, 0])
transform_win = ds_in.transform * Affine.translation(col_first, row_first)

memfile_win, ds_out_win = open_output(out_shape, transform_win)

fft_filtering_oa_strip_chain(
    ds_in=ds_in,
    ds_out=ds_out_win,
    band=1,
    kernel=kernel,
    boundary=boundary,
    out_mode=out_mode,
    win=win,
    strip_size=64,
    round_out=False,
)

filtered_win = read_back(memfile_win, ds_out_win)
filtered_win.shape

A window of 192 rows cut into strips of 64 gives three strips, so the result above went
through the overlap-add reconstruction twice. It still matches the single core call on
the same window, to the floating-point tolerance discussed in the overview.

In [ ]:
expected_win = fft_array_filter(
    mandrill_0, kernel, win=win, boundary=boundary, out_mode=out_mode
).data

np.testing.assert_allclose(filtered_win, expected_win, rtol=1e-5, atol=1e-4)

In [ ]:
plot_im({"window": mandrill_0[0:192, 160:352], "filtered": filtered_win},
        prefix="chain_002_window")

## Decimating the output

`zoom` takes the rational factor $P / Q$ of the core API, with the same restriction to
pure decimation, $P = 1$. The output then keeps one sample out of $Q$ along each axis.
`decimation` selects which sample of each block of $Q$ is kept, `"leading"` the first one
and `"centered"` the one at $\lfloor (Q - 1) / 2 \rfloor$.

```{note}
The kernel is the only anti-aliasing filter here as well. The core tutorial *Zooming out
and aliasing* shows how to check a kernel against a decimation factor, and that discussion
applies unchanged to the chain. The factor of 4 used below is the one the Gaussian of
$\sigma = 4$ handles without visible artifacts.
```

### Sizing the output

The output shape comes from the same helper as before, with the zoom and the decimation
added. It divides each axis of the window by $Q$, rounding as `decimated_size` does, so
the last sample does not necessarily land on the last row of the window.

In [ ]:
q = 4
decimation = "centered"

out_shape_dec = fft_array_filter_output_shape(
    (ds_in.height, ds_in.width),
    kernel,
    win=win,
    boundary=boundary,
    out_mode=out_mode,
    zoom=(1, q),
    decimation=decimation,
)
out_shape_dec

### Placing the output

Placing a decimated output means being explicit about two conventions.

The first one belongs to rasterio: a transform maps the upper left **corner** of a pixel,
so `transform * (0, 0)` gives the corner of the first output pixel, and pixel centers sit
at `(col + 0.5, row + 0.5)`.

The second one belongs to the decimation, which is a point sampling.
Output pixel 0 holds the full resolution sample lying `offset` samples inside the
window, with `offset = 0` for `"leading"` and `offset = (Q - 1) // 2` for `"centered"`, as
`decimation_offset` returns it. That sample covers one input pixel, whereas the output
pixel covers $Q$ of them. Placing the output pixel so that its center falls on the sample
it holds puts its corner at `offset + 0.5 - Q / 2` input pixels from the window origin.
The two terms answer different questions: `offset` is the decimation phase, and
`0.5 - Q / 2` converts a sample center into an output pixel corner.

That shift is also, read the other way, the signed distance between the kept sample and
the center of its block, the sample center sitting at `offset + 0.5` and the block center
at `Q / 2`. It is zero for an odd $Q$ in `"centered"` mode, the sample then falling on the
block center, and exactly minus half an input pixel for an even $Q$, since `(Q - 1) // 2`
rounds down. This is the bias described in the core tutorial, written into the
georeferencing rather than left implicit. An output produced with `out_mode="full"` would
need a further shift of the kernel margin, its first sample sitting ahead of the
window.

In [ ]:
def output_transform(transform_in, win, q, offset):
    """Georeferencing of a decimated output, in the frame of the input raster.

    Output pixel 0 holds the full resolution sample of index `offset` inside the
    production window. The returned transform places that output pixel so that
    its center falls on the center of that sample, which is the geometry of a
    point sampling.

    :param transform_in: affine transform of the input raster
    :param win: production window as (2, 2) inclusive (first, last) bounds
    :param q: decimation factor
    :param offset: decimation phase, as returned by `decimation_offset`
    :return: the affine transform of the decimated output
    """
    row_first, col_first = int(win[0, 0]), int(win[1, 0])
    # offset + 0.5 : center of the kept input pixel, from the window origin
    # - q / 2      : half the width of an output pixel, center to corner
    shift = offset + 0.5 - q / 2
    return (transform_in
            * Affine.translation(col_first + shift, row_first + shift)
            * Affine.scale(q, q))


offset = decimation_offset(q, decimation)
transform_dec = output_transform(ds_in.transform, win, q, offset)

offset, transform_dec

The placement can be checked rather than taken on trust. The center of output pixel
`(0, 0)`, read through the output transform, has to coincide with the center of the input
pixel holding the sample it was built from.

In [ ]:
# `offset=` here is the rasterio keyword selecting the pixel center, unrelated to
# the decimation phase held by the `offset` variable.
kept_sample = xy(ds_in.transform,
                 int(win[0, 0]) + offset, int(win[1, 0]) + offset, offset="center")
first_output = xy(transform_dec, 0, 0, offset="center")

print(f"center of the kept input sample: {kept_sample}")
print(f"center of output pixel (0, 0):   {first_output}")
np.testing.assert_allclose(first_output, kept_sample)

### What the output covers

The figure below places the output grid in the frame of the input raster, for four
settings. Blue is what the call produces: the pixels of the output dataset, the samples
they hold, the corner that `transform * (0, 0)` returns, and the extent the dataset ends
up covering. The grey blocks and their center are a reference, drawn to show where the
phase puts the kept sample inside its block. Everything is anchored on the origin of the
production window.

The window is 11 pixels wide in the figure, not a multiple of the decimation factor.

With $Q = 4$,

- in `"centered"` mode the extent overflows the window by half a pixel
on each side;
- in `"leading"` mode it starts one and a half pixels outside and stops half a
pixel short of the window end, which leaves the last requested row outside every output
pixel.

With $Q = 3$ the four samples of each axis push the extent a full pixel past the
window.

This concerns the extent declared by the transform, not the values written, but it is what
a downstream tool uses to place the product. The window used in the rest of this notebook
is 192 rows and columns, a multiple of 4, so the last sample lands on the last block and
the overflow is the symmetric half pixel of the second panel.

In [ ]:
_ = plot_output_geometry(prefix="chain_002_output_geometry")

```{note}
A raster can declare how its values relate to its grid, and that declaration belongs to
whoever produces it. GDAL carries it as the `AREA_OR_POINT` tag, `Area` meaning that a
value summarises the cell it occupies, `Point` that it is a measurement taken at the
location of the sample. A decimation keeps one sample out of $Q$, so
an output produced this way sits on the `Point` side, which
`ds_out.update_tags(AREA_OR_POINT="Point")` records in the file.

The tag itself moves nothing, GDAL keeping the geotransform on pixel corners either way.
```

### Running the decimated filtering

In [ ]:
memfile_dec, ds_out_dec = open_output(out_shape_dec, transform_dec)

fft_filtering_oa_strip_chain(
    ds_in=ds_in,
    ds_out=ds_out_dec,
    band=1,
    kernel=kernel,
    boundary=boundary,
    out_mode=out_mode,
    win=win,
    strip_size=64,
    zoom=(1, q),
    decimation=decimation,
    round_out=False,
)

filtered_dec = read_back(memfile_dec, ds_out_dec)
filtered_dec.shape

In [ ]:
plot_im(filtered_dec, prefix="chain_002_decimated")

## What's next

The output dataset has now been sized and placed for a window and for a decimation, which
covers the geometry side of the chain. The next tutorial looks at `strip_size` itself, at
what it costs in memory, and at the cases where the chain decides on its own to process
the window in a single call.